In [1]:
# --- 1. TIỀN XỬ LÝ DỮ LIỆU ---
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Đọc dữ liệu
df = pd.read_csv('adult.csv')
# Loại bỏ missing values (dạng '?')
df_clean = df.replace('?', pd.NA).dropna()

# Gom nhóm thuộc tính 'education' (16 -> 6 nhóm)
education_map = {
    'Preschool':'dropout','1st-4th':'dropout', '2nd-5th':'dropout', '6th-8th':'dropout', '9th':'dropout', '10th':'dropout', '11th':'dropout', '12th':'dropout', 'HS-grad':'HighGrad',
    'Bachelors':'Bachelors','Masters':'Masters','Doctorate':'Doctorate'
}
df_clean['education'] = df_clean['education'].map(education_map)

# Mã hóa các thuộc tính phân loại bằng LabelEncoder
label_encoders = {}
categorical_cols = ['workclass','education','marital-status','occupation',
                    'relationship','race','gender','native-country']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le

# Tách đặc trưng (X) và nhãn (y), chia tập train/test
X = df_clean.drop('income', axis=1)
y = df_clean['income'].apply(lambda x: 1 if x == '>50K' else 0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 2. XÂY DỰNG MÔ HÌNH ---
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# 2a. Decision Tree cơ bản
dt_base = DecisionTreeClassifier(random_state=42)
dt_base.fit(X_train, y_train)

# 2b. Random Forest cơ bản
rf_base = RandomForestClassifier(random_state=42, n_estimators=100)
rf_base.fit(X_train, y_train)

# --- 3. TÌM KIẾM SIÊU THAM SỐ (GRID SEARCH CHO RANDOM FOREST) ---
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42),
                           param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

# --- 4. PHÂN TÍCH ĐỘ QUAN TRỌNG ĐẶC TRƯNG ---
importances = best_rf.feature_importances_
feature_names = X_train.columns
feat_imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=False)
print(feat_imp_df.head(10))

Best Parameters: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
            feature  importance
10     capital-gain    0.147534
7      relationship    0.137900
0               age    0.121073
4   educational-num    0.111611
2            fnlwgt    0.105120
5    marital-status    0.085607
12   hours-per-week    0.075653
6        occupation    0.059345
11     capital-loss    0.044417
1         workclass    0.033957


In [1]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def evaluate_models(X_train, y_train, X_test, y_test):
    models = {
        'Perceptron': ('perceptron', Perceptron(random_state=42, max_iter=1000)),
        'Gaussian Naive Bayes': ('gaussianNB', GaussianNB()),
        'Linear SVM': ('linearSVM', SVC(kernel='linear', gamma='auto', random_state=42)),
        'Radial SVM': ('radialSVM', SVC(kernel='rbf', gamma=0.22, random_state=42)),
        'Logistic Regression': ('logisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
        'Random Forest': ('randomForest', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
        'KNN': ('kNN', KNeighborsClassifier(n_neighbors=9))
    }

    results = []
    for name, (model_name, model) in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results.append((name, acc))

    results = sorted(results, key=lambda x: x[1], reverse=True)

    best_model = results[0][0]
    best_acc = results[0][1]

    print("📊 BẢNG SO SÁNH ĐỘ CHÍNH XÁC CÁC MÔ HÌNH:")
    print("="*60)
    print("Mô hình\t\tĐộ chính xác (Test)")
    for name, acc in results:
        print(f"{name}\t\t{acc:.4f} ({acc*100:.2f}%)")
    print("="*60)

    print(f"\n🏆 MÔ HÌNH TỐT NHẤT: {best_model}")
    print(f"   Độ chính xác (Test): {best_acc:.4f} ({best_acc*100:.2f}%)")

    # Vẽ biểu đồ so sánh
    models_df = pd.DataFrame(results, columns=['Mô hình', 'Test Accuracy'])
    models_df.set_index('Mô hình', inplace=True)
    models_df.plot(kind='barh', figsize=(12, 6))
    plt.xlabel('Độ Chính Xác (Accuracy)', fontsize=12, fontweight='bold')
    plt.title('So sánh Độ Chính Xác của 7 Mô Hình Machine Learning', fontsize=14, fontweight='bold')
    plt.xlim([0.7, 0.95])
    plt.show()

evaluate_models(X_train, y_train, X_test, y_test)

NameError: name 'X_train' is not defined